<sub>**The Annotated ML Handbook** · Part VII: ML for Research · by Mahir Absar Khan</sub>

# Chapter 31: Reproducibility, leakage checklists and writing it up

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahir-606/annotated-ml-handbook/blob/main/notebooks/part-7-ml-for-research/31-research-reporting.ipynb) &nbsp; [📚 Contents](../../book/README.md) &nbsp; [🌐 Web edition](https://mahir-606.github.io/annotated-ml-handbook/)

*Run the setup cell first, then read and run the cells from top to bottom. Every code cell is followed by a table explaining what each line does.*

In [1]:
# Repository setup (not part of the book's examples): run this cell first.
import os, sys
from pathlib import Path

# Files the examples create (CSV, models, figures) go into ./outputs next to the notebook.
OUT = Path.cwd() if Path.cwd().name == "outputs" else Path.cwd() / "outputs"
OUT.mkdir(exist_ok=True)
os.chdir(OUT)

# insurance.csv is the practice dataset created in Chapter 11 and reused by later chapters.
# If it is not here yet, copy it from the repository's data/ folder (or download it in Colab).
if not Path("insurance.csv").exists():
    import shutil, urllib.request
    for base in OUT.parents:
        if (base / "data" / "insurance.csv").exists():
            shutil.copy(base / "data" / "insurance.csv", "insurance.csv")
            break
    else:
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/Mahir-606/"
                                       "annotated-ml-handbook/main/data/insurance.csv", "insurance.csv")
        except Exception as e:
            print("Could not fetch insurance.csv; run Chapter 11 first.", e)

# The figure style used in the book.
import matplotlib.pyplot as plt
from cycler import cycler
plt.rcParams.update({
    "figure.figsize": (7.2, 4.0), "figure.dpi": 100,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.28, "grid.linewidth": 0.6,
    "axes.prop_cycle": cycler(color=["#1f4fff", "#e0245e", "#0f9b62", "#ff6a00",
                                     "#7b3ff2", "#0091b3", "#d99a00", "#d42a2a"]),
    "axes.titleweight": "bold", "axes.titlesize": 12.5, "axes.titlelocation": "left",
    "font.size": 10.5, "legend.frameon": False, "image.cmap": "viridis",
})

# Plotly: interactive in Jupyter/Colab; tools/run_all.py stores static PNGs so GitHub can show them.
try:
    import plotly.io as pio
    pio.renderers.default = os.environ.get("PLOTLY_RENDERER") or (
        "colab" if "google.colab" in sys.modules else "notebook_connected")
except ImportError:
    pass

A result that cannot be reproduced cannot be trusted, and ML pipelines have many places where irreproducibility and leakage hide. This chapter gives you a project layout, the habits that make reruns exact, a leakage checklist, a reporting checklist, and a Methods-section template.

## 31.1 A project layout that scales · *Beginner*

```text
my-ml-study/
├── README.md              <- question, how to run everything, where results land
├── environment.yml        <- or requirements.txt with pinned versions
├── config/
│   └── experiment.yaml    <- seeds, paths, model choices, hyperparameter grids
├── data/
│   ├── raw/               <- original data, never edited (read-only)
│   ├── interim/           <- cleaned / merged
│   └── processed/         <- final modelling tables
├── notebooks/             <- exploration (numbered: 01_eda.ipynb, 02_baselines.ipynb)
├── src/
│   ├── data.py            <- load_and_clean()
│   ├── features.py        <- build_preprocessor()
│   ├── models.py          <- candidate pipelines
│   └── evaluate.py        <- metrics, bootstrap CIs, plots
├── scripts/
│   └── run_experiment.py  <- one command reproduces every number and figure
├── results/               <- metrics JSON/CSV, saved models (one sub-folder per run)
└── figures/               <- publication figures generated by code
```

> **💡 Tip** · **One command to rule them all**
>
> If `python scripts/run_experiment.py --config config/experiment.yaml` regenerates every table and figure in your paper, you can change anything (data, seed, model) and update the whole manuscript in minutes, and so can a reviewer.

## 31.2 A configuration-driven experiment script · *Intermediate*

**run_experiment.py in miniature**

In [2]:
import json, time, platform
from pathlib import Path
import numpy as np
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

CONFIG = {
    "seed": 2026,
    "cv": {"n_splits": 5, "n_repeats": 3},
    "scoring": ["roc_auc", "balanced_accuracy"],
    "models": {"logreg": {"C": 1.0}, "rf": {"n_estimators": 300}},
}

def build(name, params, seed):
    if name == "logreg":
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, **params))
    if name == "rf":
        return RandomForestClassifier(random_state=seed, n_jobs=-1, **params)
    raise ValueError(f"unknown model {name!r}")

def main(cfg):
    run_dir = Path("results") / time.strftime("run_%Y%m%d")
    run_dir.mkdir(parents=True, exist_ok=True)
    X, y = load_breast_cancer(return_X_y=True)
    cv = RepeatedStratifiedKFold(**cfg["cv"], random_state=cfg["seed"])
    out = {"config": cfg, "versions": {"python": platform.python_version(), "numpy": np.__version__,
                                       "sklearn": sklearn.__version__}, "results": {}}
    for name, params in cfg["models"].items():
        r = cross_validate(build(name, params, cfg["seed"]), X, y, cv=cv, scoring=cfg["scoring"], n_jobs=-1)
        out["results"][name] = {m: {"mean": round(float(r[f"test_{m}"].mean()), 4),
                                    "sd": round(float(r[f"test_{m}"].std()), 4)} for m in cfg["scoring"]}
    (run_dir / "metrics.json").write_text(json.dumps(out, indent=2))
    return out

result = main(CONFIG)
print(json.dumps(result["results"], indent=2))

{
  "logreg": {
    "roc_auc": {
      "mean": 0.9948,
      "sd": 0.0044
    },
    "balanced_accuracy": {
      "mean": 0.974,
      "sd": 0.0155
    }
  },
  "rf": {
    "roc_auc": {
      "mean": 0.9906,
      "sd": 0.0076
    },
    "balanced_accuracy": {
      "mean": 0.9554,
      "sd": 0.0218
    }
  }
}


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>import json, time, platform</code></td><td>Import the standard-library <code>json</code> module and the standard-library <code>time</code> module and the <code>platform</code> module (system and version info).</td></tr>
<tr><td><code>from pathlib import Path</code></td><td>Import <code>Path</code> from <code>pathlib</code>.</td></tr>
<tr><td><code>import numpy as np</code></td><td>Import NumPy (the array library) under the short name <code>np</code>.</td></tr>
<tr><td><code>import sklearn</code></td><td>Import scikit-learn.</td></tr>
<tr><td><code>from sklearn.datasets import load_breast_cancer</code></td><td>Import <code>load_breast_cancer</code> from scikit-learn&#x27;s <code>datasets</code> module.</td></tr>
<tr><td><code>from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate</code></td><td>Import <code>RepeatedStratifiedKFold</code> and <code>cross_validate</code> from scikit-learn&#x27;s <code>model_selection</code> module.</td></tr>
<tr><td><code>from sklearn.pipeline import make_pipeline</code></td><td>Import <code>make_pipeline</code> from scikit-learn&#x27;s <code>pipeline</code> module.</td></tr>
<tr><td><code>from sklearn.preprocessing import StandardScaler</code></td><td>Import <code>StandardScaler</code> from scikit-learn&#x27;s <code>preprocessing</code> module.</td></tr>
<tr><td><code>from sklearn.linear_model import LogisticRegression</code></td><td>Import <code>LogisticRegression</code> from scikit-learn&#x27;s <code>linear_model</code> module.</td></tr>
<tr><td><code>from sklearn.ensemble import RandomForestClassifier</code></td><td>Import <code>RandomForestClassifier</code> from scikit-learn&#x27;s <code>ensemble</code> module.</td></tr>
<tr><td><code>CONFIG = {</code></td><td>In a real project this would be read from a YAML/JSON file.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&quot;seed&quot;: 2026,</code></td><td>The single seed for the whole run.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&quot;cv&quot;: {&quot;n_splits&quot;: 5, &quot;n_repeats&quot;: 3},</code></td><td>Cross-validation settings.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&quot;scoring&quot;: [&quot;roc_auc&quot;, &quot;balanced_accuracy&quot;],</code></td><td>Metrics to compute.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&quot;models&quot;: {&quot;logreg&quot;: {&quot;C&quot;: 1.0}, &quot;rf&quot;: {&quot;n_estimators&quot;: 300}},</code></td><td>Models to run and their hyperparameters.</td></tr>
<tr><td><code>}</code></td><td>Close the dictionary opened above.</td></tr>
<tr><td><code>def build(name, params, seed):</code></td><td>Define a function <code>build</code> that takes <code>name</code>, <code>params</code> and <code>seed</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;if name == &quot;logreg&quot;:</code></td><td>Run the indented block only if <code>name == &quot;logreg&quot;</code> is true.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;return make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, **params))</code></td><td><code>**params</code> passes the config values as keyword arguments.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;if name == &quot;rf&quot;:</code></td><td>Run the indented block only if <code>name == &quot;rf&quot;</code> is true.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;return RandomForestClassifier(random_state=seed, n_jobs=-1, **params)</code></td><td>Same for the random forest.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;raise ValueError(f&quot;unknown model {name!r}&quot;)</code></td><td>Fail loudly on typos in the config.</td></tr>
<tr><td><code>def main(cfg):</code></td><td>Define a function <code>main</code> that takes <code>cfg</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;run_dir = Path(&quot;results&quot;) / time.strftime(&quot;run_%Y%m%d&quot;)</code></td><td>One folder per run.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;run_dir.mkdir(parents=True, exist_ok=True)</code></td><td>Create the folder if needed.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;X, y = load_breast_cancer(return_X_y=True)</code></td><td>Load the data.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;cv = RepeatedStratifiedKFold(**cfg[&quot;cv&quot;], random_state=cfg[&quot;seed&quot;])</code></td><td>CV splitter built from the config.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;out = {&quot;config&quot;: cfg, &quot;versions&quot;: {&quot;python&quot;: platform.python_version(), &quot;numpy&quot;: np.__version__,</code></td><td>The output record starts with the config and versions...</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&quot;sklearn&quot;: sklearn.__version__}, &quot;results&quot;: {}}</code></td><td>...and an empty results section.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;for name, params in cfg[&quot;models&quot;].items():</code></td><td>Loop over <code>cfg[&quot;models&quot;].items()</code>, naming each item <code>name, params</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;r = cross_validate(build(name, params, cfg[&quot;seed&quot;]), X, y, cv=cv, scoring=cfg[&quot;scoring&quot;], n_jobs=-1)</code></td><td>Evaluate this model.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;out[&quot;results&quot;][name] = {m: {&quot;mean&quot;: round(float(r[f&quot;test_{m}&quot;].mean()), 4),</code></td><td>Store the mean...</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&quot;sd&quot;: round(float(r[f&quot;test_{m}&quot;].std()), 4)} for m in cfg[&quot;scoring&quot;]}</code></td><td>...and SD of every metric.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;(run_dir / &quot;metrics.json&quot;).write_text(json.dumps(out, indent=2))</code></td><td>Everything needed to trace a number back to its origin.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return out</code></td><td>Send <code>out</code> back to the caller.</td></tr>
<tr><td><code>result = main(CONFIG)</code></td><td>Run the whole experiment.</td></tr>
<tr><td><code>print(json.dumps(result[&quot;results&quot;], indent=2))</code></td><td>Show the results section.</td></tr>
</table>
</details>

## 31.3 Reproducibility checklist · *Beginner*

| Item | How |
|---|---|
| Seeds | one `SEED` constant; `random_state=SEED` everywhere; `keras.utils.set_random_seed(SEED)` |
| Environment | `pip freeze > requirements.txt` or `conda env export > environment.yml`; record versions in each results file |
| Data versions | keep raw data read-only; store a checksum (`hashlib.sha256`) of each input file; note download dates |
| Code versions | Git; tag the commit that produced the submitted results (`git tag paper-v1`) |
| Configuration | hyperparameters and paths in config files, saved alongside each run |
| Determinism | report mean ± SD over several seeds rather than relying on bit-exact reruns, especially on GPUs |
| Artefacts | save fitted pipelines (`joblib`), Keras models (`.keras`), predictions, and figure data |
| Sharing | public repository (GitHub, GitLab) with an archived DOI (Zenodo); data availability statement |

**Fingerprint your data files**

In [3]:
import hashlib
Path("data.csv").write_text("a,b\n1,2\n3,4\n")
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()
print("data.csv sha256:", sha256("data.csv")[:16], "...")

data.csv sha256: b9485148546419a0 ...


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>import hashlib</code></td><td>Import the <code>hashlib</code> module (checksums).</td></tr>
<tr><td><code>Path(&quot;data.csv&quot;).write_text(&quot;a,b\n1,2\n3,4\n&quot;)</code></td><td>A stand-in data file for the demo.</td></tr>
<tr><td><code>def sha256(path, chunk=1 &lt;&lt; 20):</code></td><td>Define a function <code>sha256</code> that takes <code>path</code> and <code>chunk</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;h = hashlib.sha256()</code></td><td>A SHA-256 hash object.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;with open(path, &quot;rb&quot;) as f:</code></td><td>Open in binary mode: hash the exact bytes.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;for block in iter(lambda: f.read(chunk), b&quot;&quot;):</code></td><td>Read in 1 MB chunks so huge files don&#x27;t fill memory.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;h.update(block)</code></td><td>Feed each chunk into the hash.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return h.hexdigest()</code></td><td>Send <code>h.hexdigest()</code> back to the caller.</td></tr>
<tr><td><code>print(&quot;data.csv sha256:&quot;, sha256(&quot;data.csv&quot;)[:16], &quot;...&quot;)</code></td><td>Store this in your results; if it changes, the data changed.</td></tr>
</table>
</details>

## 31.4 The data-leakage checklist · *Beginner*

> **📖 Definitions**
>
> **Label permutation test**: re-running the whole pipeline with randomly shuffled labels. Scores should fall to chance; if they don't, something is leaking.


Leakage, meaning information from outside the training data creeping into the model, is the most common reason ML results fail to replicate. Go through this list before trusting any result.

| # | Check | Typical failure |
|---|---|---|
| 1 | Is every data-dependent preprocessing step inside a Pipeline that is fitted on training folds only? | scaling, imputation, feature selection, PCA or SMOTE done on all data |
| 2 | Is the test set used exactly once? | repeatedly checking test scores while iterating |
| 3 | Does the split respect the data's structure? | the same patient, site, animal or molecule in train and test (use group splits) |
| 4 | Does the split respect time? | training on the future to predict the past |
| 5 | Are any features proxies for the target? | "treatment given" when predicting diagnosis; IDs or timestamps correlated with label |
| 6 | Are there duplicates or near-duplicates across splits? | augmented copies, repeated records, overlapping image crops |
| 7 | Were hyperparameters or thresholds tuned on the test set? | choosing the threshold that maximises test F1 |
| 8 | Is the evaluation population the target population? | testing on a cleaner subset than real-world data |
| 9 | Does the performance look too good to be true? | near-perfect scores on a hard problem: investigate before celebrating |

> **🐞 Bug hunt** · **"My model is 99% accurate"**
>
> Before celebrating, run three checks. 1) Shuffle the labels and re-run the whole pipeline: performance must drop to chance, or something is leaking. 2) Train using only the most important feature: if it alone gives near-perfect accuracy, ask whether it could encode the answer. 3) Check for identical rows in train and test (`pd.merge(train, test, how="inner")`).

**The label-permutation sanity check**

In [4]:
from sklearn.model_selection import permutation_test_score
X, y = load_breast_cancer(return_X_y=True)
score, perm_scores, pvalue = permutation_test_score(
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)), X, y,
    scoring="roc_auc", cv=5, n_permutations=100, random_state=0, n_jobs=-1)
print(f"real AUC={score:.3f} | shuffled-label AUCs: mean={perm_scores.mean():.3f}, max={perm_scores.max():.3f} | p={pvalue:.3f}")

real AUC=0.995 | shuffled-label AUCs: mean=0.496, max=0.571 | p=0.010


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>from sklearn.model_selection import permutation_test_score</code></td><td>Import <code>permutation_test_score</code> from scikit-learn&#x27;s <code>model_selection</code> module.</td></tr>
<tr><td><code>X, y = load_breast_cancer(return_X_y=True)</code></td><td>Load the data.</td></tr>
<tr><td><code>score, perm_scores, pvalue = permutation_test_score(</code></td><td>Returns the real score, the shuffled scores and a p-value...</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)), X, y,</code></td><td>...for this model and data...</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;scoring=&quot;roc_auc&quot;, cv=5, n_permutations=100, random_state=0, n_jobs=-1)</code></td><td>Refit on 100 label shuffles to build a null distribution.</td></tr>
<tr><td><code>print(f&quot;real AUC={score:.3f} | shuffled-label AUCs: mean={perm_scores.mean():.3f}, max={perm_scores.max():.3f} | p={pvalue:.3f}&quot;)</code></td><td>Compare the real score with the null distribution.</td></tr>
</table>
</details>

The shuffled-label scores hover around 0.5 as they should. If they didn't, your pipeline would be leaking. The p-value (its smallest possible value is 1/101 with 100 permutations) says the real score is unlikely under "no relationship".

## 31.5 What to report · *Beginner*

Community reporting guidelines exist for ML-based research. **TRIPOD+AI** (2024) covers clinical prediction models, and **REFORMS** (2024) is a general checklist for ML-based science. Check whether your field has its own (for example CLAIM for medical imaging). The core items are common to all of them:

- **Data**: source, inclusion/exclusion criteria, sample size, class balance, missing data and how it was handled, time period, preprocessing.
- **Outcome/target**: exact definition, and how and by whom it was labelled.
- **Splits**: method (random, stratified, grouped, temporal), sizes, seeds, whether an external validation set exists.
- **Models**: every candidate considered (not just the winner), with software and versions.
- **Tuning**: search space, search method, selection metric, CV scheme (nested?).
- **Evaluation**: primary and secondary metrics with confidence intervals; a baseline; calibration for probabilistic predictions; performance in relevant subgroups.
- **Interpretation**: method used, its limitations, stability.
- **Limitations**: dataset shift, generalisability, potential biases.
- **Availability**: code, trained models and data (or how to request them).

## 31.6 A Methods-section template · *Beginner*

> **🔬 Research practice** · **Template: adapt and fill from your run records**
>
> **Data.** We used [dataset, source, version/date], comprising [n] [units] with [p] features. [Inclusion/exclusion criteria.] The outcome was [definition], with prevalence [x%]. Missing values ([% per variable]) were imputed with [method] fitted within training folds only.
>
> **Preprocessing and models.** Continuous features were [standardised/log-transformed]; categorical features were one-hot encoded. All preprocessing was embedded in scikit-learn pipelines (v[version]) to prevent information leakage. We compared [baseline], [model 1], [model 2], … .
>
> **Validation.** Data were split [stratified/grouped by subject/temporally] into training ([n]) and held-out test ([n]) sets (seed [s]). Model comparison and hyperparameter tuning used [k]-fold cross-validation repeated [r] times on the training set; hyperparameters were selected by [grid/random search] over [search space] to maximise [metric]. The decision threshold was fixed on out-of-fold training predictions to [criterion]. The final model was evaluated once on the test set.
>
> **Statistics.** We report [metrics] with 95% confidence intervals from [B] bootstrap resamples of the test set. Models were compared with [corrected resampled t-test / McNemar / paired bootstrap]. Feature importance was assessed by permutation importance on the test set ([n] repeats).
>
> **Software and availability.** Analyses used Python [v], NumPy [v], pandas [v], scikit-learn [v] [and TensorFlow v]. Code is available at [repository/DOI].

## 31.7 Ethics and responsible use · *Intermediate*

- **Bias and fairness**: report performance per relevant subgroup (sex, age band, site, ethnicity where appropriate). A model that is accurate overall can fail badly for a minority group. `fairlearn` provides tools for this.
- **Privacy**: de-identify data, respect consent and data-use agreements, and be aware that models can memorise individual records.
- **Intended use**: state what the model is for, and what it must *not* be used for. A research prototype is not a validated clinical or legal decision tool.
- **Honest claims**: prediction is not causation; internal validation is not external validation.

> **✍️ Try it yourself**
>
> 1. Turn the [Chapter 29](29-research-protocol.ipynb) case study into a `run_experiment.py` with a config dictionary, and save its outputs to `results/`.
> 2. Run `permutation_test_score` on the insurance regression pipeline with `scoring="r2"`.
> 3. Fill in the Methods template for your own project, marking every item you cannot yet fill: those are your to-do list.
>
> *Worked solutions:* [solutions-parts-5-8.ipynb](../../solutions/solutions-parts-5-8.ipynb), section *Chapter 31*.

---

⬅️ [Chapter 30: Choosing models and metrics, and comparing them rigorously](30-research-choosing.ipynb) · [📚 Contents](../../book/README.md) · [Chapter 32: How to debug: reading tracebacks and a systematic method](../part-8-debugging/32-debug-strategy.ipynb) ➡️